In [13]:
# 1. 라이브러리 임포트 및 유틸 함수 정의
from glob import glob
import os
import openslide
import json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy.ndimage import rotate
from skimage.transform import resize
from tqdm import tqdm
from pathlib import Path

In [14]:
# 2. 데이터 준비 및 슬라이드 매칭 함수

IHC_TYPES = ['HR', 'KI', 'ER', 'PR']

def get_sample_prefix(filename):
    """파일명에서 샘플 ID (앞 4개 파트) 추출: CODIPAI-BRCA-SS-00001"""
    basename = os.path.basename(filename)
    parts = basename.split('-')
    if len(parts) >= 4:
        return '-'.join(parts[:4])
    return basename

def get_ihc_type(filename):
    """파일명에서 IHC 타입 추출 (HR/KI/ER/PR/TP)"""
    basename = os.path.basename(filename)
    for t in IHC_TYPES + ['TP']:
        if f'-{t}-' in basename:
            return t
    return None

def match_all_ihc_pairs(data_dir):
    """
    HE(-TP-) 슬라이드를 4종 IHC(-HR-, -KI-, -ER-, -PR-)와 각각 매칭
    Returns: list of (he_path, ihc_path, ihc_type, sample_prefix)
    """
    all_files = glob(os.path.join(data_dir, '*.svs'))
    
    # 샘플별로 파일 분류
    by_sample = {}
    for f in all_files:
        prefix = get_sample_prefix(f)
        ihc_type = get_ihc_type(f)
        if prefix not in by_sample:
            by_sample[prefix] = {}
        if ihc_type:
            by_sample[prefix][ihc_type] = f
    
    pairs = []
    for prefix, files in sorted(by_sample.items()):
        if 'TP' not in files:
            print(f"  [SKIP] {prefix}: HE(TP) 없음")
            continue
        he_path = files['TP']
        for ihc_type in IHC_TYPES:
            if ihc_type in files:
                pairs.append((he_path, files[ihc_type], ihc_type, prefix))
            else:
                print(f"  [SKIP] {prefix}: {ihc_type} 없음")
    return pairs

# 사용 예시
data_dir = '../../data/Leica/Leica_Breast'
all_pairs = match_all_ihc_pairs(data_dir)
print(f"총 매칭 쌍 수: {len(all_pairs)}")
print(f"(HE 10장 x IHC 4종 = {10*4}쌍 예상)")
print()
for he_path, ihc_path, ihc_type, prefix in all_pairs[:8]:
    print(f"  {prefix}  HE: {os.path.basename(he_path)}  <->  {ihc_type}: {os.path.basename(ihc_path)}")

In [15]:
# 3. 썸네일 및 마스크 생성 함수

def get_he_thumbnail_and_mask(slide_path, downsample=30):
    slide = openslide.OpenSlide(slide_path)
    thumb = slide.get_thumbnail((slide.dimensions[0]//downsample, slide.dimensions[1]//downsample))
    thumb_np = np.array(thumb)
    gray = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2HSV)[:,:,1]
    mask = np.astype(np.where(gray>10, 255, 0), np.uint8)
    return slide, thumb_np, mask

def get_ihc_thumbnail_and_mask(slide_path, downsample=30):
    slide = openslide.OpenSlide(slide_path)
    thumb = slide.get_thumbnail((slide.dimensions[0]//downsample, slide.dimensions[1]//downsample))
    thumb_np = np.array(thumb)
    gray = cv2.cvtColor(thumb_np, cv2.COLOR_RGB2HSV)[:,:,0]
    mask = np.astype(np.where(gray>2, 255, 0), np.uint8)
    return slide, thumb_np, mask

# 예시: all_pairs의 첫 번째 쌍 (HE <-> HR)
idx = 5
he_path, ihc_path, ihc_type, prefix = all_pairs[idx]
he_slide, he_thumb_np, he_mask = get_he_thumbnail_and_mask(he_path)
ihc_slide, ihc_thumb_np, ihc_mask = get_ihc_thumbnail_and_mask(ihc_path)
print(f"Sample: {prefix}  ({ihc_type})")
print(f"HE thumbnail shape: {he_thumb_np.shape}")
print(f"IHC({ihc_type}) thumbnail shape: {ihc_thumb_np.shape}")
print(f"HE mask tissue ratio: {np.sum(he_mask > 0) / he_mask.size:.2%}")
print(f"IHC mask tissue ratio: {np.sum(ihc_mask > 0) / ihc_mask.size:.2%}")

In [16]:
# Grayscale 변환 후 Otsu threshold로 마스크 보정
he_gray = cv2.cvtColor(he_thumb_np, cv2.COLOR_RGB2GRAY)
ihc_gray = cv2.cvtColor(ihc_thumb_np, cv2.COLOR_RGB2GRAY)

_, he_mask = cv2.threshold(he_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
_, ihc_mask = cv2.threshold(ihc_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

kernel = np.ones((5,5), np.uint8)
he_mask = cv2.morphologyEx(he_mask, cv2.MORPH_CLOSE, kernel)
he_mask = cv2.morphologyEx(he_mask, cv2.MORPH_OPEN, kernel)
ihc_mask = cv2.morphologyEx(ihc_mask, cv2.MORPH_CLOSE, kernel)
ihc_mask = cv2.morphologyEx(ihc_mask, cv2.MORPH_OPEN, kernel)

print(f"HE thumbnail shape: {he_thumb_np.shape}")
print(f"IHC({ihc_type}) thumbnail shape: {ihc_thumb_np.shape}")
print(f"HE mask tissue ratio: {np.sum(he_mask > 0) / he_mask.size:.2%}")
print(f"IHC mask tissue ratio: {np.sum(ihc_mask > 0) / ihc_mask.size:.2%}")

In [17]:
# 4. 마스크 기반 정합 함수 (Grid Search)

def compute_mask_similarity(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0:
        return 0
    return intersection / union

def transform_and_match(source_mask, target_mask, angle, scale, tx, ty):
    h, w = source_mask.shape
    target_h, target_w = target_mask.shape
    rotated = rotate(source_mask, angle, reshape=False, order=0)
    new_h, new_w = int(h * scale), int(w * scale)
    scaled = cv2.resize(rotated.astype(np.uint8), (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    result = np.zeros((target_h, target_w), dtype=np.uint8)
    start_y = (target_h - scaled.shape[0]) // 2 + ty
    start_x = (target_w - scaled.shape[1]) // 2 + tx
    src_start_y = max(0, -start_y)
    src_start_x = max(0, -start_x)
    src_end_y = min(scaled.shape[0], target_h - start_y)
    src_end_x = min(scaled.shape[1], target_w - start_x)
    dst_start_y = max(0, start_y)
    dst_start_x = max(0, start_x)
    dst_end_y = dst_start_y + (src_end_y - src_start_y)
    dst_end_x = dst_start_x + (src_end_x - src_start_x)
    if src_end_y > src_start_y and src_end_x > src_start_x:
        result[dst_start_y:dst_end_y, dst_start_x:dst_end_x] = \
            scaled[src_start_y:src_end_y, src_start_x:src_end_x]
    similarity = compute_mask_similarity(result > 0, target_mask > 0)
    return similarity, result

In [18]:
# 5. 정합 최적화 (Coarse-to-Fine Grid Search)

def optimize_mask_registration(he_mask, pdl1_mask):
    # coarse search
    angle_range_coarse = np.arange(-180, 181, 10)
    scale_range_coarse = np.arange(0.8, 1.21, 0.1)
    max_translation_coarse = int(min(he_mask.shape) * 0.1)
    translation_range_coarse = np.arange(-max_translation_coarse, max_translation_coarse + 1, max(1, max_translation_coarse // 2))
    best_score, best_angle, best_scale, best_tx, best_ty, best_transformed = 0, 0, 1.0, 0, 0, None
    results = []
    total_combinations = (len(angle_range_coarse) * len(scale_range_coarse) * len(translation_range_coarse) ** 2)
    with tqdm(total=total_combinations, desc="Coarse") as pbar:
        for angle in angle_range_coarse:
            for scale in scale_range_coarse:
                for tx in translation_range_coarse:
                    for ty in translation_range_coarse:
                        score, transformed = transform_and_match(he_mask, pdl1_mask, angle, scale, tx, ty)
                        results.append((angle, scale, tx, ty, score))
                        if score > best_score:
                            best_score, best_angle, best_scale, best_tx, best_ty, best_transformed = score, angle, scale, tx, ty, transformed.copy()
                        pbar.update(1)
    # fine search (angle/scale)
    angle_range_fine = np.arange(best_angle - 10, best_angle + 11, 1)
    scale_range_fine = np.arange(max(0.5, best_scale - 0.1), min(1.5, best_scale + 0.11), 0.01)
    with tqdm(total=len(angle_range_fine) * len(scale_range_fine), desc="Fine A/S") as pbar:
        for angle in angle_range_fine:
            for scale in scale_range_fine:
                score, transformed = transform_and_match(he_mask, pdl1_mask, angle, scale, best_tx, best_ty)
                results.append((angle, scale, best_tx, best_ty, score))
                if score > best_score:
                    best_score, best_angle, best_scale, best_transformed = score, angle, scale, transformed.copy()
                pbar.update(1)
    # fine search (translation)
    translation_range_fine_x = np.arange(best_tx - 10, best_tx + 11, 1)
    translation_range_fine_y = np.arange(best_ty - 10, best_ty + 11, 1)
    with tqdm(total=len(translation_range_fine_x) * len(translation_range_fine_y), desc="Fine Trans") as pbar:
        for tx in translation_range_fine_x:
            for ty in translation_range_fine_y:
                score, transformed = transform_and_match(he_mask, pdl1_mask, best_angle, best_scale, tx, ty)
                results.append((best_angle, best_scale, tx, ty, score))
                if score > best_score:
                    best_score, best_tx, best_ty, best_transformed = score, tx, ty, transformed.copy()
                pbar.update(1)
    return best_angle, best_scale, best_tx, best_ty, best_score, best_transformed, results

In [ ]:
# 6. 정합 결과 시각화 함수

def plot_registration_results(
    he_mask, ihc_mask, best_transformed,
    he_thumb_np, ihc_thumb_np, ihc_type,
    best_angle, best_scale, best_tx, best_ty, best_score
):
    fig, axes = plt.subplots(3, 3, figsize=(24, 18))
    alpha = 0.5

    # 1행: 마스크
    axes[0, 0].imshow(he_mask, cmap='gray'); axes[0, 0].set_title('HE Mask'); axes[0, 0].axis('off')
    axes[0, 1].imshow(ihc_mask, cmap='gray'); axes[0, 1].set_title(f'{ihc_type} Mask'); axes[0, 1].axis('off')
    axes[0, 2].imshow(best_transformed, cmap='gray')
    axes[0, 2].set_title(f'HE Transformed\n(a={best_angle}°, s={best_scale:.2f}, tx={best_tx}, ty={best_ty})')
    axes[0, 2].axis('off')

    # 2행: 썸네일 + 변환 이미지
    axes[1, 0].imshow(he_thumb_np); axes[1, 0].set_title('HE Thumbnail'); axes[1, 0].axis('off')
    axes[1, 1].imshow(ihc_thumb_np); axes[1, 1].set_title(f'{ihc_type} Thumbnail'); axes[1, 1].axis('off')

    he_transformed_img = rotate(he_thumb_np, best_angle, reshape=False, order=1)
    new_h = int(he_thumb_np.shape[0] * best_scale)
    new_w = int(he_thumb_np.shape[1] * best_scale)
    he_transformed_img = cv2.resize(he_transformed_img, (new_w, new_h))
    target_h, target_w = ihc_thumb_np.shape[:2]
    result_img = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    start_y = (target_h - he_transformed_img.shape[0]) // 2 + best_ty
    start_x = (target_w - he_transformed_img.shape[1]) // 2 + best_tx
    src_sy = max(0, -start_y); src_sx = max(0, -start_x)
    src_ey = min(he_transformed_img.shape[0], target_h - start_y)
    src_ex = min(he_transformed_img.shape[1], target_w - start_x)
    dst_sy = max(0, start_y); dst_sx = max(0, start_x)
    dst_ey = dst_sy + (src_ey - src_sy); dst_ex = dst_sx + (src_ex - src_sx)
    if src_ey > src_sy and src_ex > src_sx:
        result_img[dst_sy:dst_ey, dst_sx:dst_ex] = he_transformed_img[src_sy:src_ey, src_sx:src_ex]
    axes[1, 2].imshow(result_img); axes[1, 2].set_title('HE Transformed Image'); axes[1, 2].axis('off')

    # 3행: Overlay
    overlay = np.zeros((*ihc_mask.shape, 3), dtype=np.uint8)
    overlay[best_transformed > 0] = [255, 0, 0]
    overlay[ihc_mask > 0] += np.array([0, 0, 255], dtype=np.uint8)
    axes[2, 0].imshow(overlay)
    axes[2, 0].set_title(f'Mask Overlay (IoU={best_score:.4f})\nRed=HE, Blue={ihc_type}')
    axes[2, 0].axis('off')

    he_resized = cv2.resize(he_thumb_np, (target_w, target_h))
    axes[2, 1].imshow(cv2.addWeighted(he_resized, alpha, ihc_thumb_np, 1-alpha, 0))
    axes[2, 1].set_title('Before Registration'); axes[2, 1].axis('off')

    axes[2, 2].imshow(cv2.addWeighted(result_img, alpha, ihc_thumb_np, 1-alpha, 0))
    axes[2, 2].set_title('After Registration'); axes[2, 2].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# 7. 변환 파라미터 계산 및 저장 함수

def get_transform_matrix(angle, scale, tx, ty, center_x, center_y):
    angle_rad = np.radians(angle)
    cos_a = np.cos(angle_rad) * scale
    sin_a = np.sin(angle_rad) * scale
    M = np.array([
        [cos_a, -sin_a, -center_x * cos_a + center_y * sin_a + center_x + tx],
        [sin_a, cos_a, -center_x * sin_a - center_y * cos_a + center_y + ty],
        [0, 0, 1]
    ])
    return M

def save_transformation_params(he_slide_path, ihc_slide_path, he_slide, ihc_slide,
                               he_thumb_shape, ihc_thumb_shape,
                               angle, scale, tx, ty, score, ihc_type,
                               output_dir='../../data/Leica/registration_params'):
    output_dir = Path(output_dir)
    output_dir.mkdir(exist_ok=True, parents=True)
    prefix = get_sample_prefix(he_slide_path)
    output_file = output_dir / f"{prefix}_{ihc_type}_registration.json"
    downsample_x = he_slide.dimensions[0] / he_thumb_shape[1]
    downsample_y = he_slide.dimensions[1] / he_thumb_shape[0]
    tx_fullres = tx * downsample_x
    ty_fullres = ty * downsample_y
    he_center_x = he_thumb_shape[1] / 2
    he_center_y = he_thumb_shape[0] / 2
    M_thumbnail = get_transform_matrix(angle, scale, tx, ty, he_center_x, he_center_y)
    he_center_x_full = he_slide.dimensions[0] / 2
    he_center_y_full = he_slide.dimensions[1] / 2
    M_fullres = get_transform_matrix(angle, scale, tx_fullres, ty_fullres, he_center_x_full, he_center_y_full)
    data = {
        'files': {
            'he_slide': str(he_slide_path),
            'ihc_slide': str(ihc_slide_path),
            'ihc_type': ihc_type,
            'prefix': prefix
        },
        'dimensions': {
            'he_full': {'width': he_slide.dimensions[0], 'height': he_slide.dimensions[1]},
            'ihc_full': {'width': ihc_slide.dimensions[0], 'height': ihc_slide.dimensions[1]},
            'he_thumbnail': {'width': he_thumb_shape[1], 'height': he_thumb_shape[0]},
            'ihc_thumbnail': {'width': ihc_thumb_shape[1], 'height': ihc_thumb_shape[0]}
        },
        'transformation': {
            'thumbnail': {
                'angle_degrees': float(angle),
                'scale': float(scale),
                'translation_x': float(tx),
                'translation_y': float(ty),
                'matrix': M_thumbnail[:2, :].tolist()
            },
            'fullres': {
                'angle_degrees': float(angle),
                'scale': float(scale),
                'translation_x': float(tx_fullres),
                'translation_y': float(ty_fullres),
                'matrix': M_fullres[:2, :].tolist()
            }
        },
        'downsampling': {
            'factor_x': float(downsample_x),
            'factor_y': float(downsample_y)
        },
        'quality': {
            'iou_score': float(score)
        }
    }
    with open(output_file, 'w') as f:
        json.dump(data, f, indent=2)
    print(f"  Saved: {output_file.name}")
    return output_file


In [ ]:
# 8. 모든 쌍에 대해 정합 및 파라미터 저장 (HE 1장 x IHC 4종)

def process_all_slide_pairs(all_pairs,
                            output_dir='../../data/Leica/registration_params',
                            vis_dir='../../data/Leica/registration_vis'):
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(vis_dir, exist_ok=True)
    total = len(all_pairs)
    for idx, (he_path, ihc_path, ihc_type, prefix) in enumerate(all_pairs):
        print(f"\n[{idx+1}/{total}] {prefix}  HE <-> {ihc_type}")
        print(f"  HE : {os.path.basename(he_path)}")
        print(f"  IHC: {os.path.basename(ihc_path)}")
        he_slide, he_thumb_np, he_mask = get_he_thumbnail_and_mask(he_path)
        ihc_slide, ihc_thumb_np, ihc_mask = get_ihc_thumbnail_and_mask(ihc_path)

        # Otsu 마스크 보정
        he_gray = cv2.cvtColor(he_thumb_np, cv2.COLOR_RGB2GRAY)
        ihc_gray = cv2.cvtColor(ihc_thumb_np, cv2.COLOR_RGB2GRAY)
        _, he_mask = cv2.threshold(he_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        _, ihc_mask = cv2.threshold(ihc_gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel = np.ones((5, 5), np.uint8)
        he_mask = cv2.morphologyEx(he_mask, cv2.MORPH_CLOSE, kernel)
        he_mask = cv2.morphologyEx(he_mask, cv2.MORPH_OPEN, kernel)
        ihc_mask = cv2.morphologyEx(ihc_mask, cv2.MORPH_CLOSE, kernel)
        ihc_mask = cv2.morphologyEx(ihc_mask, cv2.MORPH_OPEN, kernel)

        best_angle, best_scale, best_tx, best_ty, best_score, best_transformed, _ = \
            optimize_mask_registration(he_mask, ihc_mask)

        save_transformation_params(
            he_path, ihc_path, he_slide, ihc_slide,
            he_thumb_np.shape, ihc_thumb_np.shape,
            best_angle, best_scale, best_tx, best_ty, best_score,
            ihc_type=ihc_type, output_dir=output_dir
        )

        # 시각화 저장
        fig, axes = plt.subplots(3, 3, figsize=(24, 18))
        axes[0, 0].imshow(he_mask, cmap='gray'); axes[0, 0].set_title('HE Mask'); axes[0, 0].axis('off')
        axes[0, 1].imshow(ihc_mask, cmap='gray'); axes[0, 1].set_title(f'{ihc_type} Mask'); axes[0, 1].axis('off')
        axes[0, 2].imshow(best_transformed, cmap='gray')
        axes[0, 2].set_title(f'HE Transformed\n(a={best_angle}°, s={best_scale:.2f}, tx={best_tx}, ty={best_ty})')
        axes[0, 2].axis('off')

        overlay = np.zeros((*ihc_mask.shape, 3), dtype=np.uint8)
        overlay[best_transformed > 0] = [255, 0, 0]
        overlay[ihc_mask > 0] += np.array([0, 0, 255], dtype=np.uint8)
        axes[1, 0].imshow(he_thumb_np); axes[1, 0].set_title('HE Thumbnail'); axes[1, 0].axis('off')
        axes[1, 1].imshow(ihc_thumb_np); axes[1, 1].set_title(f'{ihc_type} Thumbnail'); axes[1, 1].axis('off')

        # HE 이미지 변환
        he_transformed_img = rotate(he_thumb_np, best_angle, reshape=False, order=1)
        new_h = int(he_thumb_np.shape[0] * best_scale)
        new_w = int(he_thumb_np.shape[1] * best_scale)
        he_transformed_img = cv2.resize(he_transformed_img, (new_w, new_h))
        target_h, target_w = ihc_thumb_np.shape[:2]
        result_img = np.zeros((target_h, target_w, 3), dtype=np.uint8)
        start_y = (target_h - he_transformed_img.shape[0]) // 2 + best_ty
        start_x = (target_w - he_transformed_img.shape[1]) // 2 + best_tx
        src_sy = max(0, -start_y); src_sx = max(0, -start_x)
        src_ey = min(he_transformed_img.shape[0], target_h - start_y)
        src_ex = min(he_transformed_img.shape[1], target_w - start_x)
        dst_sy = max(0, start_y); dst_sx = max(0, start_x)
        dst_ey = dst_sy + (src_ey - src_sy); dst_ex = dst_sx + (src_ex - src_sx)
        if src_ey > src_sy and src_ex > src_sx:
            result_img[dst_sy:dst_ey, dst_sx:dst_ex] = he_transformed_img[src_sy:src_ey, src_sx:src_ex]

        axes[1, 2].imshow(result_img); axes[1, 2].set_title('HE Transformed Image'); axes[1, 2].axis('off')

        axes[2, 0].imshow(overlay)
        axes[2, 0].set_title(f'Mask Overlay (IoU={best_score:.4f})\nRed=HE, Blue={ihc_type}, Purple=Both')
        axes[2, 0].axis('off')

        alpha = 0.5
        he_resized = cv2.resize(he_thumb_np, (target_w, target_h))
        axes[2, 1].imshow(cv2.addWeighted(he_resized, alpha, ihc_thumb_np, 1-alpha, 0))
        axes[2, 1].set_title('Before Registration'); axes[2, 1].axis('off')

        axes[2, 2].imshow(cv2.addWeighted(result_img, alpha, ihc_thumb_np, 1-alpha, 0))
        axes[2, 2].set_title('After Registration'); axes[2, 2].axis('off')

        plt.tight_layout()
        vis_path = os.path.join(vis_dir, f'{prefix}_{ihc_type}_registration_vis.png')
        plt.savefig(vis_path)
        plt.close(fig)
        print(f"  IoU={best_score:.4f}  vis saved: {os.path.basename(vis_path)}")

    print(f"\n완료: 총 {total}쌍 정합 처리")

# 실행
all_pairs = match_all_ihc_pairs('../../data/Leica/Leica_Breast')
process_all_slide_pairs(all_pairs)

In [25]:
# registration_vis 폴더에서 페어가 되지 않는 파일 json 삭제
def clean_unmatched_json_files(param_dir='../../data/Leica/registration_params', vis_dir='../../data/Leica/registration_vis'):
    param_files = glob(os.path.join(param_dir, '*.json'))
    vis_files = glob(os.path.join(vis_dir, '*.png'))
    vis_basenames = set(os.path.basename(f).replace('_registration_vis.png', '') for f in vis_files)
    removed_count = 0
    for param_file in param_files:
        basename = os.path.basename(param_file).replace('_registration.json', '')
        if basename not in vis_basenames:
            os.remove(param_file)
            removed_count += 1
            print(f"Removed unmatched JSON: {os.path.basename(param_file)}")
    print(f"Total unmatched JSON files removed: {removed_count}")

clean_unmatched_json_files()


Removed unmatched JSON: CODIPAI-BRCA-SS-00198_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00001_PR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00193_HR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00191_ER_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00001_ER_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00192_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00196_HR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00193_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00195_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00001_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00191_HR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00198_HR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00191_KI_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00194_HR_registration.json
Removed unmatched JSON: CODIPAI-BRCA-SS-00192_HR_registration.

In [ ]:
# 8. JSON 불러오기 및 좌표 변환 함수

def load_and_apply_transformation(json_path):
    with open(json_path, 'r') as f:
        params = json.load(f)
    print(f"Loaded registration parameters from: {json_path}")
    print(f"  Prefix: {params['files']['prefix']}")
    print(f"  HE slide: {Path(params['files']['he_slide']).name}")
    print(f"  PDL1 slide: {Path(params['files']['pdl1_slide']).name}")
    print(f"  Transformation: angle={params['transformation']['fullres']['angle_degrees']}°, "
          f"scale={params['transformation']['fullres']['scale']:.2f}, "
          f"tx={params['transformation']['fullres']['translation_x']:.1f}, "
          f"ty={params['transformation']['fullres']['translation_y']:.1f}")
    print(f"  Quality: IoU={params['quality']['iou_score']:.4f}")
    return params

def transform_patch_coordinates(he_x, he_y, patch_size, params):
    M = np.array(params['transformation']['fullres']['matrix'])
    corners = np.array([
        [he_x, he_y, 1],
        [he_x + patch_size, he_y, 1],
        [he_x + patch_size, he_y + patch_size, 1],
        [he_x, he_y + patch_size, 1]
    ]).T
    transformed = M @ corners
    center_x = transformed[0].mean()
    center_y = transformed[1].mean()
    pdl1_x = int(center_x - patch_size / 2)
    pdl1_y = int(center_y - patch_size / 2)
    return pdl1_x, pdl1_y, transformed.T

# 사용 예시
print("\n=== Example: How to use saved parameters for patch extraction ===\n")
print("# 1. Load JSON parameters")
print("params = load_and_apply_transformation('registration_results/CODIPAI-XXXX_registration.json')")
print("")
print("# 2. Extract aligned patches")
print("he_x, he_y = 10000, 20000  # HE 패치 좌표")
print("patch_size = 512")
print("pdl1_x, pdl1_y, corners = transform_patch_coordinates(he_x, he_y, patch_size, params)")
print("")
print("# 3. Read patches from WSI")
print("he_patch = he_slide.read_region((he_x, he_y), 0, (patch_size, patch_size))")
print("pdl1_patch = pdl1_slide.read_region((pdl1_x, pdl1_y), 0, (patch_size, patch_size))")